# Neural Collaborative Filtering avec PyTorch

Le réseau, le negative sampling, l'entraînement, l'évaluation et les recommandations sont définis dans `src/recommend_deep_learning.py`. Ce notebook montre uniquement le flux d'utilisation.

In [217]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")
sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend_deep_learning as ncf_module
ncf_module = importlib.reload(ncf_module)

## 1. Charger les données et choisir le matériel

In [218]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)
DEVICE = "auto"

pd.Series({
    "dataset_version": listens["dataset_version"].iloc[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
    "mps_available": torch.backends.mps.is_available(),
    "cuda_available": torch.cuda.is_available(),
})

dataset_version      clean-v1.0.0-19b966482277
interactions                             10064
users                                       82
artist_identities                         2640
mps_available                             True
cuda_available                           False
dtype: object

## 2. Sélectionner la configuration et entraîner le réseau final

Trois configurations d'embeddings et de régularisation sont comparées sur la validation. La meilleure est réentraînée sur train + validation.

In [219]:
import src.recommend_deep_learning as ncf_module
ncf_module = importlib.reload(ncf_module)

experiment = ncf_module.select_and_fit_best_ncf(
    listens,
    epochs=5,
    top_n=10,
    random_state=42,
    device=DEVICE,
    verbose=True,
)

display(experiment.tuning_results)
display(pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
}))
print(f"Dataset : {experiment.dataset_version}")
print(f"Meilleure configuration : {experiment.best_config}")

NCF 1/3 — embedding_8
  époque 1/5 — loss=0.5797
  époque 2/5 — loss=0.5632
  époque 3/5 — loss=0.5603
  époque 4/5 — loss=0.5582
  époque 5/5 — loss=0.5550
  Recall@10=0.0374 | NDCG@10=0.0280
NCF 2/3 — embedding_16
  époque 1/5 — loss=0.5947
  époque 2/5 — loss=0.5601
  époque 3/5 — loss=0.5549
  époque 4/5 — loss=0.5502
  époque 5/5 — loss=0.5447
  Recall@10=0.0193 | NDCG@10=0.0146
NCF 3/3 — embedding_8_regularized
  époque 1/5 — loss=0.5800
  époque 2/5 — loss=0.5647
  époque 3/5 — loss=0.5630
  époque 4/5 — loss=0.5617
  époque 5/5 — loss=0.5574
  Recall@10=0.0557 | NDCG@10=0.0646
  époque 1/5 — loss=0.5836
  époque 2/5 — loss=0.5640
  époque 3/5 — loss=0.5630
  époque 4/5 — loss=0.5614
  époque 5/5 — loss=0.5584
Meilleure configuration : embedding_8_regularized


,config,embedding_dim,weight_decay,final_loss,recall@10,ndcg@10,n_evaluable_users
0,embedding_8,8,0.0000,0.554952,0.037427,0.028048,67
1,embedding_16,16,0.0000,0.544650,0.019267,0.014562,67
2,embedding_8_regularized,8,0.0001,0.557421,0.055734,0.064567,67


,validation,test_final
recall@10,0.055734,0.033123
ndcg@10,0.064567,0.024989
n_evaluable_users,67.000000,66.000000


Dataset : clean-v1.0.0-19b966482277
Meilleure configuration : embedding_8_regularized


## 3. Vérifier la convergence du modèle final

In [220]:
pd.DataFrame({
    "epoch": range(1, len(experiment.recommender.training_losses_) + 1),
    "training_loss": experiment.recommender.training_losses_,
})

,epoch,training_loss
0,1,0.583639
1,2,0.563995
2,3,0.562952
3,4,0.561354
4,5,0.558430


## 4. Obtenir 10 recommandations

In [221]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)

,rank,artist_uid,artist,score
0,1,mbid:d997b0d8-6954-4dfb-9bb2-ebb6d8a4c29c,Piruka,0.380060
1,2,mbid:c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.371324
2,3,mbid:d867b44b-6662-46e7-a8fb-11008eb4b5b2,Masicka,0.370822
3,4,mbid:97b8d224-5660-419c-8dff-4b3bf0c33983,The Neighbourhood,0.364936
4,5,mbid:d6cd2165-7877-4ca7-ace9-34dacadaf393,Dishwalla,0.362162
5,6,mbid:9fff2f8a-21e6-47de-a2b8-7f449929d43f,Drake,0.361051
6,7,mbid:75be165a-ad83-4d12-bd28-f589a15c479f,Major Lazer,0.359027
7,8,mbid:48262e82-db9f-4a92-b650-dfef979b73ec,Future,0.358517
8,9,mbid:373a4c98-a46b-48e4-86ec-f6ca65b4f438,Chance the Rapper,0.358246
9,10,mbid:6c2a79b8-2b5c-4c2d-b495-6a446811bf9d,Malcolm Todd,0.349570
